
# PS2: Mini-batch SGD and CNN

Advanced Learning 2026/7.



For SUBMISSION:   

Please upload the complete and executed `ipynb` to your git repository. Verify that all of your output can be viewed directly from github, and provide a link to that git file below.

~~~
STUDENT ID: 208781641
~~~

~~~
STUDENT GIT LINK: https://github.com/sharon2897-queen/SharonsHomeWork/blob/main/PS2_Part1_CNN_2026_ID_208781641%20(1).ipynb
~~~
In Addition, don't forget to add your ID to the files, and upload to moodle the html version:    
  
`PS2_CNN_RNN_2026_ID_[208781641].html`   




---


Imports:

In [1]:
import numpy as np # You are allowed to use  only numpy.
import time, sys, cProfile
import matplotlib.pyplot as plt

Please import the objects in `srcPS2.py` file into the workspace

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


In [ ]:
your_drive_path_to_src_file = "/content/drive/MyDrive/[PATH TO YOUR DIRECTORY WHERE SRC IS IN]]"
sys.path.append(your_drive_path_to_src_file)

In [6]:
from srcPS2 import *

Note that now all of the network's objects from PS1 are loaded into the workspace.

**Alternatively**, you can upload srcPS2.py in the files tab found on the left hand side without needing to mount your drive, and import as is. DONE!

---

### 0. Loading data

As in PS1, you will evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.  



In [2]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


---

## PS2 Part 1: Mini-batch SGD



**Welcome back**.   






Mini-batch Stochastic Gradient Descent (Mini-batch SGD) is an optimization algorithm used to train machine learning models, particularly neural networks. It strikes a balance between two other methods:

    Batch Gradient Descent – Uses the entire dataset to compute gradients and update the model parameters.
        ✅ Accurate gradients
        ❌ Slow and memory-intensive for large datasets

    Stochastic Gradient Descent (SGD) – Uses a single data point to compute gradients for each update.
        ✅ Fast and efficient
        ❌ Noisy updates, which can lead to instability

  
**Mini-batch SGD**.

Mini-batch SGD computes gradients and updates the model parameters using small, randomly selected batches of data (typically 32, 64, or 128 samples).

    ✅ Faster than batch gradient descent
    ✅ Less noisy than pure SGD
    ✅ Efficient use of memory
    ✅ Enables vectorization for faster computation on GPUs

**How it Works**.

    Shuffle the dataset to avoid bias.

    Divide the data into small batches (e.g., 64 samples per batch).

    For each epoch, iterate through the mini-batches:
        Perform forward pass to calculate predictions.
        Compute the loss for the mini-batch.
        Perform backpropagation to compute gradients.
        Update model parameters using the gradients.

    Repeat until the model converges.

**Algorithm**

For each mini-batch $B={x_1,x_2,...,x_m}B={x_1​,x_2​,...,x_m​}$ of size $m$, update the weights $w$ as:    

$$
w=w−η⋅\frac{1}{m}∑_{i=1}^m ∇L(w,x_i)
$$

Where:   
*  $η$ = learning rate.
* $∇L(w,x_i)$ = gradient of the loss for sample $x_i$.   

Advantages

    Efficient and scalable – Works well with large datasets.
    Smooth convergence – Less noisy than SGD but faster than full batch training.
    Parallelization – Batches can be processed in parallel on GPUs.

Disadvantages

    Choosing batch size can affect performance. Small batches lead to noise, large batches consume more memory.
    Convergence may be slower than batch gradient descent for some problems.


### Mini-batch Questions.
**P1Q1**:  

Please answer the following questions providing statistical justifications and mathematical formulas when needed.

P1Q1a: Explain how the mini-batch size impacts convergence, model performance, and computational efficiency.



The mini-batch size m governs a fundamental trade-off between **gradient variance**, **generalization**, and **hardware efficiency**:

1. **Convergence:**
   The mini-batch gradient estimator is $\hat{g}_m = \frac{1}{m}\sum_{i=1}^{m} \nabla L(w, x_i)$. Assuming i.i.d. sampling, $\hat{g}_m$ is an unbiased estimator of the true gradient $\nabla L(w)$, with covariance:
   $$\text{Var}(\hat{g}_m) = \frac{\Sigma}{m}$$
   where $\Sigma = \text{Cov}(\nabla L(w, x))$. Smaller $m$ yields higher variance (noisier updates, oscillating around minima), whereas larger $m$ reduces variance by a factor of $1/m$ and gives smoother, more accurate descent steps at the cost of fewer parameter updates per epoch.

2. **Model Performance :**
   Small mini-batches act as an implicit regularizer: gradient noise helps the optimizer escape sharp local minima and settle into flat minima, which empirically generalize better on unseen data. Very large batches often converge to sharp minima.

3. **Computational Efficiency:**
   Larger mini-batches enable vectorized matrix-matrix operations, maximizing throughput (samples/sec) up to memory limits. Very small batches underutilize hardware parallelism and incur high loop overhead per sample.

**P1Q1b:**   
Why does mini-batch SGD require a learning rate schedule, and what are the common scheduling strategies?



In mini-batch SGD, the gradient estimate retains non-zero variance $\text{Var}(\hat{g}_m) = \frac{\Sigma}{m}$ even at a stationary point where $\nabla L(w^*) = 0$. With a constant learning rate $\eta$, updates satisfy $w_{t+1} = w_t - \eta \hat{g}_m$, causing the parameters to bounce endlessly in a noise ball around $w^*$ rather than converging.
By the Robbins Monro conditions for stochastic approximation, convergence to a stationary point requires a decaying learning rate $\eta_t$ such that:
$$\sum_{t=1}^{\infty} \eta_t = \infty \quad \text{and} \quad \sum_{t=1}^{\infty} \eta_t^2 < \infty$$
Large initial $\eta$ enables fast exploration and escaping poor local optima, while smaller $\eta$ later allows fine-grained convergence.

**Common scheduling strategies:**
* **Step Decay:** Multiply $\eta$ by a factor $\gamma < 1$ every $k$ epochs: $\eta_t = \eta_0 \gamma^{\lfloor t/k \rfloor}$.
* **Exponential Decay:** $\eta_t = \eta_0 e^{-\lambda t}$ (or $\eta_0 \gamma^t$).
* **Cosine Annealing:** $\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T}\pi\right)\right)$.
* **Warmup + Decay:** Linearly increase $\eta$ for the first few epochs to stabilize early gradients, then decay.
* **ReduceLROnPlateau:** Reduce $\eta$ dynamically when validation loss stops improving.

P1Q1c: What role does batch normalization play in mini-batch SGD, and why is it essential for deep neural networks?



Batch Normalization normalizes the pre-activation values of each layer across the current mini-batch $B$ of size $m$:
$$\mu_B = \frac{1}{m}\sum_{i=1}^m x_i, \qquad \sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2$$
$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad y_i = \gamma \hat{x}_i + \beta$$
where $\gamma, \beta$ are learnable scale and shift parameters preserving representational capacity.

**Why it is essential for deep neural networks:**
* **Smoother optimization landscape:** BN bounds the Lipschitz constant of the loss and its gradients, reducing gradient explosion/vanishing and allowing significantly higher learning rates.
* **Reduced sensitivity to initialization & scale drift:** prevents activations from saturating nonlinearities (like tanh/sigmoid) across deep layers.
* **Mild regularization:** Because $\mu_B$ and $\sigma_B^2$ are noisy estimates computed per mini-batch, BN injects slight stochasticity that improves generalization.

P1Q1d: How does mini-batch SGD differ in performance compared to full-batch gradient descent when dealing with non-convex optimization problems?



In non-convex optimization:

* **Full-batch GD** follows the exact negative gradient $-\nabla L(w)$. Near a saddle point (where $\nabla L(w) = 0$ but the Hessian has negative eigenvalues) or in a sharp local minimum, full-batch GD slows down drastically or gets trapped permanently.
* **Mini-batch SGD** introduces stochastic noise $\epsilon_t = \hat{g}_m - \nabla L(w)$ with $\mathbb{E}[\epsilon_t]=0$ and $\text{Cov}(\epsilon_t)=\Sigma/m$. This structured noise perturbs the trajectory along directions of negative curvature, allowing mini-batch SGD to escape saddle points and narrow/sharp local minima much more effectively and converge toward wider, flatter basins that generalize better.

P1Q1e: How does mini-batch SGD perform when training on imbalanced datasets, and what strategies can mitigate its challenges?



On imbalanced datasets, uniform random sampling produces mini-batches dominated by majority class examples. Consequently, the mini-batch gradient $\hat{g}_m$ is heavily biased toward minimizing majority-class error, and minority-class samples may be absent from many batches entirely causing poor recall on minority classes and unstable minority gradients.

**Mitigation strategies:**
* **Stratified mini-batch Sampling (Oversampling minority / Undersampling majority):** Enforce a balanced class ratio inside each mini-batch.
* **Class-Weighted Loss / Focal Loss:** Scale each sample's loss by inverse class frequency $w_c \propto \frac{1}{N_c}$ (or use Focal Loss $-(1-p_t)^\gamma \log(p_t)$ to focus gradients on hard/minority examples).
* **Data Augmentation :** Generate synthetic minority-class examples to enrich minority representation without simple duplication.

P1Q1f: Explain why larger mini-batch sizes are preferred during inference but not necessarily during training.


* **During Inference:** No gradients, activations for backpropagation, or optimizer states need to be stored in memory, and there is no optimization/generalization concern about sharp vs. flat minima because weights are frozen. Therefore, larger mini-batches strictly improve throughput (samples/sec) by maximizing vectorized matrix-matrix multiplication (BLAS/GPU utilization) and amortizing data-transfer overhead.
* **During Training:** Memory consumption scales linearly with batch size (storing intermediate activations for backprop), and excessively large batches reduce gradient noise $\text{Var}(\hat{g}_m) \propto 1/m$, which reduces the number of weight updates per epoch and often hurts generalization by converging to sharper minima.

### Applying mini-batch SGD

Preparing the data:

In [3]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

P1Q2a:  

Add a new function to the class `MyNetwork` named `fit_mini_batch`. This function should be similar to `fit`, just with a mini-batch SGD implementation. You can also make other code changes to the scripts in `srcPS2` if you think they are needed. In addition, add a suitable learning rate scheduler of your choice.


YOU SHOULD (a) add the missing script to the srcPS2 code, and (b) make a copy of the function  below for review:

In [7]:
def fit_mini_batch(self, x_train, y_train, batch_size, epochs, learning_rate):
    n = len(x_train)
    print(f"Training on {n:,} samples (batch_size={batch_size}):")

    for ep in range(epochs):
        lr = learning_rate(ep) if callable(learning_rate) else learning_rate

        idx = np.random.permutation(n)
        x_shuf, y_shuf = x_train[idx], y_train[idx]

        total_err = 0.0
        for start in range(0, n, batch_size):
            xb = x_shuf[start:start + batch_size]
            yb = y_shuf[start:start + batch_size]

            if xb.ndim == 3 and xb.shape[1] == 1:
                xb = xb.reshape(len(xb), -1)
            if yb.ndim == 3 and yb.shape[1] == 1:
                yb = yb.reshape(len(yb), -1)

            out = xb
            for layer in self.layers:
                out = layer.forward_propagation(out)

            total_err += self.loss(yb, out) * len(xb)
            grad = self.loss_grad(yb, out)
            for layer in reversed(self.layers):
                grad = layer.backward_propagation(grad, lr)

        print(f"Training epoch {ep + 1}/{epochs}   lr={lr:.4f}   error={total_err / n:.6f}")

MyNetwork.fit_mini_batch = fit_mini_batch


    Test the two networks below to compare the differences between GD and mini-batch SGD:

In [8]:
#### GD implementaiton ####

# Network Architecture
netGD = MyNetwork()

netGD.add(Affine_Layer(28*28, 128))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(128, 64))
netGD.add(ActivationLayer(tanh, tanh_grad))
netGD.add(Affine_Layer(64, 10))
netGD.add(ActivationLayer(tanh, tanh_grad))

netGD.use_loss(mse, mse_grad)

epoch_num = 10
lr = 0.05
t1 = time.time()
netGD.fit(x_train[:10000], y_train[:10000], epochs=epoch_num, learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")

Training on 10,000 samples:
Training epoch 1/10   error=0.040442
Training epoch 2/10   error=0.023067
Training epoch 3/10   error=0.018038
Training epoch 4/10   error=0.015270
Training epoch 5/10   error=0.013388
Training epoch 6/10   error=0.011960
Training epoch 7/10   error=0.010819
Training epoch 8/10   error=0.009871
Training epoch 9/10   error=0.009065
Training epoch 10/10   error=0.008367
Total process time: 25.874


In [9]:
output_GD = netGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0121, Accuracy: 9391/10000 (94%)



In [12]:
#### mini-batch SGD implementaiton ####

netMiniGD = MyNetwork()

netMiniGD.add(Affine_Layer(28*28, 128))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(128, 64))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))
netMiniGD.add(Affine_Layer(64, 10))
netMiniGD.add(ActivationLayer(tanh, tanh_grad))

netMiniGD.use_loss(mse, mse_grad)

epoch_num = 10
lr_sched = lambda epoch: 0.8 * (0.9 ** epoch)
t2 = time.time()
netMiniGD.fit_mini_batch(x_train[:10000], y_train[:10000], batch_size=128, epochs=epoch_num, learning_rate=lr_sched)
print(f"Total process time: {round(time.time() - t2, 3)}")

Training on 10,000 samples (batch_size=128):
Training epoch 1/10   lr=0.8000   error=0.072948
Training epoch 2/10   lr=0.7200   error=0.044654
Training epoch 3/10   lr=0.6480   error=0.039830
Training epoch 4/10   lr=0.5832   error=0.036677
Training epoch 5/10   lr=0.5249   error=0.034236
Training epoch 6/10   lr=0.4724   error=0.032276
Training epoch 7/10   lr=0.4252   error=0.030761
Training epoch 8/10   lr=0.3826   error=0.029604
Training epoch 9/10   lr=0.3444   error=0.028599
Training epoch 10/10   lr=0.3099   error=0.027802
Total process time: 3.216


In [13]:
outputMiniGSD = netMiniGD.predict(x_test ,y_test )


Test set: Avg. loss: 0.0303, Accuracy: 8956/10000 (90%)



P1Q2b:   
Please answer the following:
* Which implementation was faster?
* with lower training error?
* with better test accuracy?    

and explain how these results fit your answers in Q1A.

* **Faster implementation:** `netMiniGD` was ~8x faster in wall-clock time (`3.216s` vs. `25.874s`). `netGD.fit()` iterates sample by sample in Python (SGD with `batch_size=1`), whereas `fit_mini_batch` processes 128 samples at once using vectorized NumPy matrix multiplications.
* **Lower training error:** `netGD` reached a lower training error (`0.008367` vs. `0.027802`). Over 10 epochs, `netGD` performs 10,000 weight updates per epoch (100,000 total), while `netMiniGD` performs only 79 updates per epoch (790 total). Even with the exponential LR scheduler, 790 total steps leave the training error slightly higher.
* **Better test accuracy:** `netGD` achieved `94%` test accuracy compared to `90%` for `netMiniGD`, again reflecting 128x more gradient steps over the 10 epochs.
* **Relation to P1Q1a:** This matches the trade-off from P1Q1a: mini-batches provide a large speedup through vectorization and lower gradient variance, while requiring a higher learning rate or more epochs to compensate for fewer parameter updates per epoch.

## PS2 Part 2: CNN



A Convolutional Neural Network (CNN) is a type of a deep neural network that can perform well at image recognition and analysis tasks.


> In this problem set we will add CNN compatibility to our numpy-based neural network from PS 1.

The main components needed to implement a CNN architecture:  

**Convolution layer:** In the convolutional layers, filters are applied to the image to identify patterns and features. Imagine a sliding window that moves across the image, detecting edges, shapes, and other building blocks.  

**Pooling layer:** Pooling layers downsample the data, reducing its complexity and computational cost. This helps prevent overfitting, where the model memorizes training data instead of learning generalizable patterns.  

**Reshape layer:**
Reshaping the output of the convolution and pooling layers into the fully connected layer.   


A USEFUL HINT:

*Keeping account of the input and output shapes along the CNN network is crucial for the model to work as intended*



Useful resource:  
* [CS231](https://cs231n.github.io/convolutional-networks/).
* [Convolutional Neural Network From Scratch](https://www.kaggle.com/code/lusfernandotorres/convolutional-neural-network-from-scratch).
* [CNN with PyTorch](https://www.kaggle.com/code/sdelecourt/cnn-with-pytorch-for-mnist)

#### Data transformations





(Here I added a reshape that I originally intended for you to apply at model time).

In [14]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [15]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.astype('float32')
x_train /= 255
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1],x_train.shape[2])
# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
y_train = y_train.reshape(y_train.shape[0], 1, y_train.shape[1])

# same for test data : 10000 samples
x_test = x_test.astype('float32')
x_test /= 255
x_test = x_test.reshape(x_test.shape[0],1, x_test.shape[1],x_test.shape[2])
y_test = to_categorical(y_test)
y_test = y_test.reshape(y_test.shape[0],1, y_test.shape[1])

In [16]:
print("x_train shape",x_train.shape)
print("x_test shape",x_test.shape)
print("y_train shape",y_train.shape)
print("y_test shape",y_test.shape)

x_train shape (60000, 1, 28, 28)
x_test shape (10000, 1, 28, 28)
y_train shape (60000, 1, 10)
y_test shape (10000, 1, 10)


Unlike the multilayer perceptron (MLP) architecture, the input to the convolution net is not flattened. In our case, the size of a single `MNIST` datapoint is (1,28,28). The first dimension is the number of channels (e.g. RGB  for color images). During training the inputs and outputs often  change shape as they pass through the convolution, pooling, and the other layers of the network.

---

### A. Convolution Layer

Here you are asked to implement a basic Convolution layer with the following components:



1.   Layer initialization - initialize the layer's filters using Normalized Xavier/Hu weights
2.   A generator function that yields all the available sliding windows in the input image.
3.   Forward propogation for the convolution layer
4.   Backward propogation for the convolution layer

[CS231](https://cs231n.github.io/convolutional-networks/#conv) provides a great overview both mathematically and visually,   
and [CS230](https://cs230.stanford.edu/section/4/) provides a good overview about Xavier initialization.




P2Q1a:  (10pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.    
You can deviate slightly from the recommended structure below, but please remain true to the class structure and compatibility with the rest of the network classes and functions. The idea here is to write YOUR OWN version of CNN, so please try not to google the answers.





In [17]:
class Convolution:
    def __init__(self, num_filters=3, size=3, activation=None, in_channels=1):
        self.stride = 1
        self.size = size
        self.activation = activation
        self.last_input = None

        # Normalized Xavier initialization
        fan_in = in_channels * size * size
        fan_out = num_filters * size * size
        limit = np.sqrt(6.0 / (fan_in + fan_out))
        self.filters = np.random.uniform(-limit, limit, (num_filters, in_channels, size, size))

    def patch_generator(self, image):
        ch, h, w = image.shape
        for y in range(0, h - self.size + 1, self.stride):
            for x in range(0, w - self.size + 1, self.stride):
                patch = image[:, y:y + self.size, x:x + self.size]
                yield patch, y, x

    def forward_propagation(self, image):
        if len(image.shape) == 2:
            image = np.expand_dims(image, axis=0)
        self.last_input = image

        _, h, w = image.shape
        out_h = (h - self.size) // self.stride + 1
        out_w = (w - self.size) // self.stride + 1
        output = np.zeros((self.filters.shape[0], out_h, out_w))

        for f in range(self.filters.shape[0]):
            for patch, y, x in self.patch_generator(image):
                output[f, y, x] = np.sum(patch * self.filters[f])

        if self.activation is not None:
            output = self.activation.forward_propagation(output)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        if self.activation is not None:
            grad_out = self.activation.backward_propagation(grad_out, learning_rate)

        grad_in = np.zeros(self.last_input.shape)
        grad_filter = np.zeros(self.filters.shape)

        for f in range(self.filters.shape[0]):
            for patch, y, x in self.patch_generator(self.last_input):
                g = grad_out[f, y, x]
                grad_filter[f] += patch * g
                grad_in[:, y:y + self.size, x:x + self.size] += self.filters[f] * g

        self.filters -= learning_rate * grad_filter
        return grad_in

### B. Max-pooling Layer

As with the convolution layer, here you are asked to implement a basic max-pooling layer with the following components:


1.   A generator function that yields all the available sliding windows in the input image. You can reuse the generator from (1).
2.   Forward propogation for the max-pooling layer
3.   Backward propogation for the max-pooling layer

The pooling layer is responsible for reducing the dimensionality of the input. As with the convolution layer, the pooling layer slides a filter across the entire image input to calculate the output. Note that this layer does not have any weights.  As the filter slides through the input image, it selects the pixel with the maximum value to populate the output array.

P2Q1b: (10pts).       

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [18]:
class MaxPool:
    def __init__(self, size=2):
        self.last_input = None
        self.stride = 1
        self.size = size

    def patch_generator(self, image):
        ch, h, w = image.shape
        for c in range(ch):
            for y in range(0, h - self.size + 1, self.stride):
                for x in range(0, w - self.size + 1, self.stride):
                    patch = image[c, y:y + self.size, x:x + self.size]
                    yield patch, c, y, x

    def forward_propagation(self, image):
        self.last_input = image
        num_channels, h_prev, w_prev = image.shape

        h_new = (h_prev - self.size) // self.stride + 1
        w_new = (w_prev - self.size) // self.stride + 1
        output = np.zeros((num_channels, h_new, w_new))

        for patch, c, y, x in self.patch_generator(image):
            output[c, y, x] = np.max(patch)
        return output

    def backward_propagation(self, grad_out, learning_rate):
        grad_in = np.zeros(self.last_input.shape)

        for patch, c, y, x in self.patch_generator(self.last_input):
            max_y, max_x = np.unravel_index(np.nanargmax(patch), patch.shape)
            grad_in[c, y + max_y, x + max_x] += grad_out[c, y, x]

        return grad_in

### C. Reshape Layer

A reshape layer in a neural network is a transformation layer that doesn't perform any computations itself (no weights).   
Its purpose is to simply change the dimensionality (shape) of the data flowing through the network.

Here's how reshape layers work:

You define a target shape for the output of the reshape layer.   
This target shape specifies the number of elements along each dimension (width, height, channels, etc.) for the new data format.

P2Q1c:  (5pts).   
Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the class works properly.  





In [19]:
class Reshape:
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape

    def forward_propagation(self, input_data):
        return np.reshape(input_data, self.output_shape)

    def backward_propagation(self, output_gradient, learning_rate):
        return np.reshape(output_gradient, self.input_shape)

### Testing Your Neural Network

### Defining our main neural network architecture

Define your CNN network's architecture:

* A convolution layer with some filters `(f)` and a window size of `(w x w)`, with a `tanh` activation function.
* A max pooling layer of a window size of `(p x p)`.  
* First affine layer that takes your input and outputs 128 nodes
* `tanh` activation layer following the first affine layer
* Second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh` activation layer following the second affine layer
* Third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh` activation layer following the last affine layer


You can compute the shapes of the convolution and max-pool layers' outputs  as a function of the input shape, the convolution sliding window size, and the number of filters for applied. ( the stride and padding are usually also used to calculate the output shapes, but in this simple implementation we set the stride to 1 with zero padding.).

P2Q2a: (5pts).        
Calculate the input and output shapes of each layer in the network below, up until the first affine layer (including).  (mathematically) explain your answer.  

With padding $P=0$ and stride $S=1$, a sliding window of size $K \times K$ maps spatial dimensions $(H, W)$ to $(H - K + 1, W - K + 1)$.

Tracing a single MNIST image through the network:
1. **Convolution (`num_filters=12, size=3`):**
   Input: $(1, 28, 28) \rightarrow$ Output: $(12, 28-3+1, 28-3+1) = (12, 26, 26)$
2. **MaxPool (`size=3, stride=1`):**
   Input: $(12, 26, 26) \rightarrow$ Output: $(12, 26-3+1, 26-3+1) = (12, 24, 24)$
3. **Reshape:**
   Input: $(12, 24, 24) \rightarrow$ Output: $(1, 12 \cdot 24 \cdot 24) = (1, 6912)$
4. **First Affine Layer (`6912 -> 128`):**
   Input: $(1, 6912) \rightarrow$ Output: $(1, 128)$

In [20]:
h_in, w_in = 28, 28
conv_filters, conv_k = 12, 3
pool_k = 3

conv_out = (conv_filters, h_in - conv_k + 1, w_in - conv_k + 1)
pool_out = (conv_filters, conv_out[1] - pool_k + 1, conv_out[2] - pool_k + 1)
flat_dim = pool_out[0] * pool_out[1] * pool_out[2]

print("After Conv:", conv_out)
print("After MaxPool:", pool_out)
print("After Reshape:", (1, flat_dim))
print("After Affine 1:", (1, 128))

After Conv: (12, 26, 26)
After MaxPool: (12, 24, 24)
After Reshape: (1, 6912)
After Affine 1: (1, 128)


P2Q2b:  (5pts).   

Please complete the missing code (`## SOLUTION REQUIRED!!`) so that the network architecture works properly.  





In [21]:
CNNnet = MyNetwork()
num_classes = 10

c = 12
x_new = 24
y_new = 24

CNNnet.add(Convolution(num_filters=12, size=3,
                       activation=ActivationLayer(relu, relu_grad)))
CNNnet.add(MaxPool(size=3))

CNNnet.add(Reshape((c, y_new, x_new), (1, c * y_new * x_new)))

CNNnet.add(Affine_Layer(c * y_new * x_new, 128))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(128, 64))
CNNnet.add(ActivationLayer(tanh, tanh_grad))
CNNnet.add(Affine_Layer(64, num_classes))
CNNnet.add(ActivationLayer(tanh, tanh_grad))

### Training!

There is no need to train the model on the entire dataset (you'l see why now...). Try to prove that your network works as expected by showing that your error decreases dramatically even on a small sample (say, around 1k).

In [22]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# Training will be pretty slow if we update at each iteration on 60000 samples...]
CNNnet.use_loss(mse, mse_grad)
epoch_num = 20 #20
lr = 0.1
t1 = time.time()
CNNnet.fit(x_train[:1000], y_train[:1000], epochs=epoch_num,
           learning_rate=lr)
print(f"Total process time: {round(time.time() - t1,3)}")


Training on 1,000 samples:
Training epoch 1/20   error=0.072153
Training epoch 2/20   error=0.041642
Training epoch 3/20   error=0.031543
Training epoch 4/20   error=0.025153
Training epoch 5/20   error=0.020479
Training epoch 6/20   error=0.016996
Training epoch 7/20   error=0.014375
Training epoch 8/20   error=0.012379
Training epoch 9/20   error=0.010796
Training epoch 10/20   error=0.009450
Training epoch 11/20   error=0.008254
Training epoch 12/20   error=0.007231
Training epoch 13/20   error=0.006363
Training epoch 14/20   error=0.005621
Training epoch 15/20   error=0.004972
Training epoch 16/20   error=0.004438
Training epoch 17/20   error=0.003967
Training epoch 18/20   error=0.003572
Training epoch 19/20   error=0.003244
Training epoch 20/20   error=0.002957
Total process time: 5892.083


Hurray - if you got to this point, your CNN model works, but is EXTREMELY slow.   
Let's check why using a cProfiler:

In [23]:
# CNNnet.prof(x_train[:100], y_train[:100],epochs=4,learning_rate=lr)


P2Q2c:  (5pts).     
Why do you think our CNN implementation is slow? Your answer should relate to the  approximate `O(n)` time/computational complexity .

Our CNN is slow because both convolution and max-pooling run nested Python loops over every filter and every spatial $(y, x)$ position, processing one image at a time in both the forward and backward passes.

For $E$ epochs, $N$ samples, $F$ filters, $C$ input channels, image size $H \times W$, and kernel size $K \times K$, the time complexity is:
$$O\left(E \cdot N \cdot F \cdot C \cdot H \cdot W \cdot K^2\right)$$
Even with $F=12$ and $H \approx W \approx 26$, every single image executes over 8,000 Python generator iterations in `Convolution` and ~6,900 in `MaxPool` (for both forward and backward). Without vectorizing patches (`im2col`/GEMM) or batching, Python interpreter overhead dominates runtime.

### Evaluation

Exciting! Now is the time to test your model.     


    May the gradients be always in your favor.

P2Q2b (cont'd):

**Try to limit the number of epochs and the size of the trained data, while still achieving an accuracy score of >85%.**

In [24]:
output = CNNnet.predict(x_test[:500] ,y_test[:500] )


Test set: Avg. loss: 0.0211, Accuracy: 452/500 (90%)



### Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

**Prepare the data as tensors using PyTorch DataLoader:**

In [26]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# # training data : 60000 samples
# # reshape and normalize input data
# x_train = x_train.reshape(x_train.shape[0], 1, 28*28) not needed
x_train = x_train.astype('float32')
x_train /= 255
# # One-hot encoding of the output.
# # Currently a number in range [0,9]; Change into a vector of size 10
# # e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)
# # same for test data : 10000 samples
# x_test = x_test.reshape(x_test.shape[0], 1, 28*28) not needed
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)

In [27]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchCNN` class with an identical architecture you used in your home-made network.  
Hint: use `x = x.view()` to handle reshapes.

P2Q3a: (5pts).   

In [28]:
class PyTorchCNN(nn.Module):
    def __init__(self):
        super(PyTorchCNN, self).__init__()
        num_classes = 10
        self.conv = nn.Conv2d(in_channels=1, out_channels=12, kernel_size=3, stride=1)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=1)
        self.fc1 = nn.Linear(12 * 24 * 24, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.view(-1, 1, 28, 28)
        x = F.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(-1, 12 * 24 * 24)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        return x

In [29]:

# Train the model
num_epochs = 10
pt_learning_rate = 0.01
pt_network = PyTorchCNN()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        # labels = labels.view(64,1,10)
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')


Epoch [1/10], Step [500/938], Loss: 0.0088
Epoch [2/10], Step [500/938], Loss: 0.0105
Epoch [3/10], Step [500/938], Loss: 0.0044
Epoch [4/10], Step [500/938], Loss: 0.0171
Epoch [5/10], Step [500/938], Loss: 0.0026
Epoch [6/10], Step [500/938], Loss: 0.0070
Epoch [7/10], Step [500/938], Loss: 0.0045
Epoch [8/10], Step [500/938], Loss: 0.0119
Epoch [9/10], Step [500/938], Loss: 0.0113
Epoch [10/10], Step [500/938], Loss: 0.0089


Evaluation:

In [30]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target,)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0001, Accuracy: 9707/10000 (97%)



P2Q3b (5pts).   

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

Write your solutions here:
1. **Which model performed better? Why?**
   The PyTorch model achieved much higher test accuracy (97% on all 10,000 test images vs. 90% on 500 images for the NumPy model). PyTorch trained on all 60,000 samples using mini-batches of 64 and the Adam optimizer (adaptive learning rates with momentum), whereas our NumPy CNN only trained on 1,000 samples with plain SGD.

2. **Which model performed faster? Why?**
   PyTorch was orders of magnitude faster because it runs batched operations in compiled C++/MKL/CUDA kernels (using optimized `im2col`/GEMM or cuDNN routines) instead of Python loops over single images and patches.

3. **What would you change in the network's architecture?**
   * Set `stride=2` (e.g., `kernel_size=2, stride=2`) in `MaxPool` so it actually downsamples the feature maps (from 26x26 to 13x13), drastically reducing parameters in the first linear layer and preventing overfitting.
   * Replace `tanh` in the hidden affine layers with ReLU to avoid vanishing gradients, and remove `tanh` from the output layer.
   * Add a second Conv+Pool block, Batch Normalization, and Dropout before the fully connected layers.

4. **What would you change in the solution algorithm?**
   * Replace MSE loss with Cross-Entropy loss (`nn.CrossEntropyLoss`), which is much better suited for multi-class classification and gives stronger gradients when confident predictions are wrong.
   * Add a learning rate scheduler and weight decay (L2 regularization), and vectorize the NumPy convolution (e.g., via `im2col` + `fit_mini_batch`).

In [1]:
import json, subprocess
from google.colab import _message, files

filename = f"PS2_CNN_RNN_2026_ID_{208781641}"

nb = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
nb.get("metadata", {}).pop("widgets", None)

with open(f"{filename}.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f)


subprocess.run(["jupyter", "nbconvert", "--to", "html", f"{filename}.ipynb"], check=True)
files.download(f"{filename}.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>